# LexIA — Phase 3 : Embedding, Indexing & RAG Core

Ce notebook retrace la phase d'embedding et de construction du RAG core de LexIA.
Il couvre :
- Le choix du modèle d'embedding (BGE-M3 tenté → contraintre RAM → paraphrase-multilingual)
- La construction de l'index Chroma
- La visualisation des embeddings (t-SNE)
- L'analyse du seuil MIN_RELEVANCE_SCORE
- Le test du RAG core complet
- L'analyse des réponses générées

**Input** : 27 781 chunks (`chunks.jsonl`)  
**Output** : Index Chroma + RAG core fonctionnel

## 1. Choix du modèle d'embedding

### BGE-M3 — le choix idéal

BGE-M3 (BAAI) supporte 3 types de retrieval en un seul modèle :
- **Dense** : recherche sémantique (vecteur de 1024 dims)
- **Sparse** : recherche par mots-clés style BM25
- **Multi-vector** : ColBERT style (un vecteur par token)

Pour un corpus juridique, le hybrid dense+sparse est idéal :
- Dense : trouve `"licenciement économique"` quand l'utilisateur écrit `"se faire virer"`
- Sparse : trouve `"L1233-4"` quand l'utilisateur tape le numéro exact de l'article

### Contrainte RAM — OOM documenté

```
RAM disponible  : 3.2 Go (sur 8.3 Go total, 4.7 Go utilisés par le système)
BGE-M3 complet  : 4.56 Go → OOM Kill
BGE-M3 INT8     : 2.27 Go → OOM Kill (RAM insuffisante au chargement)
```

### Modèle retenu : paraphrase-multilingual-mpnet-base-v2

| Paramètre | Valeur |
|---|---|
| Taille | ~280 Mo |
| Dimensions | 768 |
| Langues | 50+ dont français |
| Type | Dense uniquement |
| RAM utilisée | ~500 Mo |

**Migration production** : sur une machine 16 Go+, remplacer par BGE-M3 pour le hybrid retrieval dense+sparse.

## 2. Imports et configuration

In [ ]:
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from langchain_core.documents import Document

# Ajoute le projet au path
sys.path.insert(0, '/workspaces/lexia')

CHUNKS_PATH = '/workspaces/lexia/data/processed/chunks.jsonl'
INDEX_PATH  = '/workspaces/lexia/data/index/chroma'
MODEL_NAME  = 'paraphrase-multilingual-mpnet-base-v2'

print('Imports OK')

## 3. Chargement du modèle et test RAM

In [ ]:
import psutil

def ram_info():
    m = psutil.virtual_memory()
    return f"{m.available/1e9:.1f}Go dispo / {m.total/1e9:.1f}Go total"

print(f'RAM avant chargement : {ram_info()}')
model = SentenceTransformer(MODEL_NAME)
print(f'RAM après chargement : {ram_info()}')
print(f'Modèle chargé : {MODEL_NAME}')

# Test rapide
test_emb = model.encode(['test'], normalize_embeddings=True)
print(f'Dimensions embedding : {test_emb.shape[1]}')

## 4. Chargement de l'index Chroma

In [ ]:
client = chromadb.PersistentClient(
    path=INDEX_PATH,
    settings=Settings(anonymized_telemetry=False)
)
collection = client.get_collection('lexia_chunks')

print(f'Index chargé : {collection.count()} chunks')

# Stats par code
sample = collection.get(limit=2000, include=['metadatas'])
by_code = Counter(m.get('code_name', 'unknown') for m in sample['metadatas'])
by_type = Counter(m.get('chunk_type', 'unknown') for m in sample['metadatas'])

print(f'\nRépartition par code (échantillon 2000) :')
for code, count in by_code.items():
    print(f'  {code} : {count}')

print(f'\nRépartition par type de chunk :')
for ctype, count in by_type.items():
    print(f'  {ctype} : {count}')

## 5. Visualisation des embeddings — t-SNE

La visualisation t-SNE permet de vérifier que les embeddings capturent bien
la structure sémantique du corpus — les articles juridiquement proches
devraient apparaître proches dans l'espace 2D.

In [ ]:
from sklearn.manifold import TSNE

# Échantillon de 500 chunks pour la visualisation
sample_viz = collection.get(
    limit=500,
    include=['embeddings', 'metadatas']
)

embeddings_array = np.array(sample_viz['embeddings'])
codes = [m.get('code_name', 'unknown') for m in sample_viz['metadatas']]
chunk_types = [m.get('chunk_type', 'unknown') for m in sample_viz['metadatas']]

print(f'Shape embeddings : {embeddings_array.shape}')
print('Calcul t-SNE (peut prendre 30-60s)...')

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(embeddings_array)

print('t-SNE terminé')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Visualisation t-SNE des embeddings LexIA (500 chunks)', fontsize=13)

# Coloré par code juridique
ax1 = axes[0]
code_colors = {
    'Code du travail': '#1D9E75',
    'Code de la consommation': '#7F77DD',
}
for code, color in code_colors.items():
    mask = [c == code for c in codes]
    x = embeddings_2d[mask, 0]
    y = embeddings_2d[mask, 1]
    ax1.scatter(x, y, c=color, label=code, alpha=0.6, s=15)
ax1.set_title('Par code juridique')
ax1.legend(fontsize=9)
ax1.set_xlabel('t-SNE dim 1')
ax1.set_ylabel('t-SNE dim 2')
ax1.grid(alpha=0.3)

# Coloré par type de chunk
ax2 = axes[1]
type_colors = {
    'short':    '#9FE1CB',
    'standard': '#1D9E75',
    'annexe':   '#085041',
}
for ctype, color in type_colors.items():
    mask = [c == ctype for c in chunk_types]
    if any(mask):
        x = embeddings_2d[mask, 0]
        y = embeddings_2d[mask, 1]
        ax2.scatter(x, y, c=color, label=ctype, alpha=0.6, s=15)
ax2.set_title('Par type de chunk')
ax2.legend(fontsize=9)
ax2.set_xlabel('t-SNE dim 1')
ax2.set_ylabel('t-SNE dim 2')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/workspaces/lexia/notebooks/embeddings_tsne.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphique sauvegardé → notebooks/embeddings_tsne.png')

## 6. Analyse du seuil MIN_RELEVANCE_SCORE

Le seuil est un hyperparamètre critique :
- **Trop bas** : des chunks hors sujet passent (météo Bretagne → articles sur intempéries)
- **Trop haut** : des questions légitimes ne trouvent pas de chunks (rupture conventionnelle filtrée)

On mesure empiriquement le nombre de chunks retenus selon le seuil
sur nos questions de test.

In [ ]:
from indexing.vector_store import similarity_search

# Questions de test avec label (pertinente / hors sujet)
test_questions = [
    ("Mon employeur peut-il me licencier pendant un arrêt maladie ?",       True,  None),
    ("Quel est le délai de rétractation pour un achat en ligne ?",           True,  "Code de la consommation"),
    ("Quelles sont les conditions pour une rupture conventionnelle ?",        True,  "Code du travail"),
    ("Quelles sont mes droits en cas de harcèlement au travail ?",           True,  "Code du travail"),
    ("Quel est le temps qu'il fait en Bretagne ?",                           False, None),
    ("Quelle est la recette de la tarte tatin ?",                            False, None),
]

thresholds = [0.45, 0.50, 0.55, 0.58, 0.60, 0.65, 0.70]

print(f'{"Question":<55} {"Pertinente":<12}', end='')
for t in thresholds:
    print(f'  {t}', end='')
print()
print('-' * 120)

results_by_threshold = {t: {'tp': 0, 'fp': 0, 'fn': 0} for t in thresholds}

for question, is_relevant, filter_code in test_questions:
    docs = similarity_search(question, n_results=5, filter_code=filter_code)
    scores = [d.metadata.get('relevance_score', 0) for d in docs]
    
    print(f'{question[:53]:<55} {str(is_relevant):<12}', end='')
    for t in thresholds:
        count = sum(1 for s in scores if s >= t)
        print(f'  {count:>4}', end='')
        if is_relevant and count == 0:
            results_by_threshold[t]['fn'] += 1
        elif not is_relevant and count > 0:
            results_by_threshold[t]['fp'] += 1
    print()

print()
print(f'{"Seuil":<8} {"Faux positifs":<15} {"Faux négatifs":<15} {"Verdict"}')
print('-' * 60)
for t in thresholds:
    fp = results_by_threshold[t]['fp']
    fn = results_by_threshold[t]['fn']
    verdict = '✅ retenu' if t == 0.58 else ''
    print(f'{t:<8} {fp:<15} {fn:<15} {verdict}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

fps = [results_by_threshold[t]['fp'] for t in thresholds]
fns = [results_by_threshold[t]['fn'] for t in thresholds]

x = range(len(thresholds))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], fps, width,
               label='Faux positifs (hors sujet passent)', color='#D85A30', alpha=0.8)
bars2 = ax.bar([i + width/2 for i in x], fns, width,
               label='Faux négatifs (pertinents filtrés)', color='#7F77DD', alpha=0.8)

# Marque le seuil retenu
idx_retenu = thresholds.index(0.58)
ax.axvline(x=idx_retenu, color='#1D9E75', linestyle='--', linewidth=2,
           label='Seuil retenu (0.58)')

ax.set_xticks(list(x))
ax.set_xticklabels([str(t) for t in thresholds])
ax.set_xlabel('Seuil MIN_RELEVANCE_SCORE')
ax.set_ylabel('Nombre de questions mal traitées')
ax.set_title('Trade-off précision/rappel selon le seuil de pertinence')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(max(fps), max(fns)) + 1)

plt.tight_layout()
plt.savefig('/workspaces/lexia/notebooks/score_threshold.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note : optimisation rigoureuse via RAGAS en phase 5 sur dataset de référence')

## 7. Test du retrieval sur questions juridiques

In [ ]:
from indexing.vector_store import similarity_search

MIN_SCORE = 0.58

queries = [
    ("Mon employeur peut-il me licencier pendant un arrêt maladie ?", None),
    ("Quel est le délai de rétractation pour un achat en ligne ?",    "Code de la consommation"),
    ("Rupture conventionnelle — conditions et délais",                 "Code du travail"),
    ("Harcèlement moral au travail — que faire ?",                    "Code du travail"),
]

for query, filter_code in queries:
    docs = similarity_search(query, n_results=5, filter_code=filter_code)
    docs_filtered = [d for d in docs if d.metadata.get('relevance_score', 0) >= MIN_SCORE]
    
    print(f'\nQ: {query}')
    if filter_code:
        print(f'   Filtre : {filter_code}')
    print(f'   {len(docs_filtered)}/{len(docs)} chunks retenus (seuil={MIN_SCORE})')
    
    for doc in docs_filtered:
        print(f"   [{doc.metadata.get('relevance_score', 0):.3f}] "
              f"Art. {doc.metadata.get('article_num', '?'):15} "
              f"— {doc.page_content[:80]}...")

## 8. Test RAG core complet — réponses générées

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from rag.chain import ask

print('RAG core initialisé')

In [ ]:
print('=' * 60)
print('Q1 : Licenciement pendant arrêt maladie')
print('=' * 60)
response = ask(
    "Mon employeur peut-il me licencier pendant un arrêt maladie ?",
    stream=False
)
print(response)

In [ ]:
print('=' * 60)
print('Q2 : Rétractation achat en ligne')
print('=' * 60)
response = ask(
    "Quel est le délai de rétractation pour un achat en ligne ?",
    filter_code="Code de la consommation",
    stream=False
)
print(response)

In [ ]:
print('=' * 60)
print('Q3 : Rupture conventionnelle')
print('=' * 60)
response = ask(
    "Quelles sont les conditions pour une rupture conventionnelle ?",
    filter_code="Code du travail",
    stream=False
)
print(response)

In [ ]:
print('=' * 60)
print('Q4 : Hors sujet — météo Bretagne')
print('=' * 60)
response = ask(
    "Quel est le temps qu'il fait en Bretagne ?",
    stream=False
)
print(response)
print()
print('✅ Le prompt strict fonctionne — refus de répondre hors sujet')

## 9. Architecture du RAG core

```
Question utilisateur
        │
        │ rag/retriever.py
        │ • encode la question (SentenceTransformer)
        │ • similarity_search dans Chroma
        │ • filtre min_score=0.58
        │ • parent-child : renvoie article complet
        ▼
Top-K chunks pertinents
        │
        │ rag/prompt.py
        │ • format_context() — structure les articles
        │ • ChatPromptTemplate — system + human
        │ • prompt strict — grounding juridique
        ▼
Prompt enrichi
        │
        │ rag/chain.py
        │ • ChatGroq — llama-3.3-70b-versatile
        │ • temperature=0.1 (réponses factuelles)
        │ • streaming SSE disponible
        ▼
Réponse sourcée avec articles cités + URLs Légifrance
```

## 10. Conclusions et limitations

### Ce qui fonctionne
- Retrieval pertinent sur les questions du droit du travail (scores > 0.75)
- Réponses sourcées avec numéros d'articles et URLs Légifrance
- Prompt strict : refus explicite sur les questions hors sujet
- Parent-child retrieval : contexte complet envoyé au LLM

### Limitations documentées

| Limitation | Cause | Solution production |
|---|---|---|
| Dense retrieval uniquement | BGE-M3 trop lourd (RAM) | BGE-M3 sur machine 16 Go+ |
| Troncature 1500 chars | Limite TPM Groq 12k tokens | Modèle sans limite stricte |
| MIN_SCORE empirique | Pas de dataset d'éval | Optimisation RAGAS phase 5 |
| Corpus figé au 13/07/2025 | Dump statique | Pipeline mise à jour automatique |

